# Linguistic processing with spaCy

📌 **Additional material.** We do not work through this notebook in the taught sessions. It brings together, in one place, the material that used to run across four separate notebooks.

Read it if you need **linguistic** structure from your texts — sentences, lemmas, parts of speech, or named entities such as people and places. That last one, **named entity recognition**, is the reason most humanities projects reach for spaCy.

## Why linguistic processing at all?

Everything we have done this week treated a text as a bag of word-forms. That took us a long way, but it has obvious blind spots:

* `walk`, `walks`, `walked` and `walking` are counted as four unrelated words
* we cannot ask "which **verbs** appear in this corpus?" — we have no idea which words are verbs
* we cannot ask "which **people** are mentioned?" — the model has never heard of people

Linguistic processing adds those layers of analysis on top of the raw characters. The usual steps are:

1. **Sentence segmentation** — where does each sentence begin and end?
2. **Tokenisation** — splitting into words
3. **Lemmatisation** — reducing each word to its dictionary form
4. **Part-of-speech tagging** — noun, verb, adjective...
5. **Dependency parsing** — who did what to whom
6. **Named entity recognition** — people, places, organisations

### These steps are harder than they look

**Sentence segmentation.** "Split on full stops" fails immediately:

```
Input:  Mr. Smith and Mr. Jones went to the shops.

Naive output:      Mr.
                   Smith and Mr.
                   Jones went to the shops.
```

**Tokenisation.** In notebook 2a you split on whitespace, and saw that `times,` and `times` became different words. But punctuation is not always separable: what should happen to `don't`, `New York`, `sixty-four`, or `£4.50`?

This is why these steps are done by **statistical models trained on annotated text**, not by rules — and why they can be wrong, especially on historical or OCR'd material, which looks nothing like the modern news text they were trained on.

## Getting started

spaCy needs a **pipeline** for the language you are working in. We use `en_core_web_sm`, trained on modern English. There are [models for many other languages](https://spacy.io/usage/models).

In [ ]:
import spacy

# You only need to download a pipeline once:
spacy.cli.download("en_core_web_sm")

In [ ]:
# Load the pipeline into a variable, conventionally called `nlp`:
nlp = spacy.load("en_core_web_sm")

Now run a text through it. The result *looks* like the original string, but it is not:

In [ ]:
example = "This is a great week. Is it not?"
output = nlp(example)

print(output)
print(type(output))

`output` is a spaCy `Doc`: a sequence of `Token` objects, each carrying the annotations the pipeline produced. Access a token by position, and its analyses through attributes ending in an underscore:

In [ ]:
token = output[3]

print("text: ", token.text)
print("lemma:", token.lemma_)
print("POS:  ", token.pos_)

## Iterating over the annotations

A `Doc` can be looped over like a list, so everything you know about `for` loops and list comprehensions applies.

In [ ]:
for token in output:
    print(token.text, "|", token.lemma_, "|", token.pos_)

In [ ]:
# The same thing as a list comprehension:
print([token.lemma_ for token in output])

### Sentence segmentation

In [ ]:
text = "Mr. Smith and Mr. Jones went to the shops. They bought a notebook."
doc = nlp(text)

for sentence in doc.sents:
    print(sentence)

Note that it did **not** break after `Mr.` — which is exactly the failure we saw above.

### Selecting by part of speech

Because a list comprehension can carry a condition, we can pull out just the words of one kind:

In [ ]:
text = "Mr. Smith and Mr. Jones went to the shops and they bought a beautiful old notebook."
doc = nlp(text)

print("nouns:     ", [token.text for token in doc if token.pos_ == "NOUN"])
print("verbs:     ", [token.text for token in doc if token.pos_ == "VERB"])
print("adjectives:", [token.text for token in doc if token.pos_ == "ADJ"])

### ✏️ Exercise 1

Write a function called `get_adjectives` that takes a text and an `nlp` pipeline, and returns the adjectives it contains as a list. Try it on:

* `"It was a bright cold day in April, and the clocks were striking thirteen."`
* `"It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife."`

In [ ]:
# Type your code here:


### Dependency parsing

Dependency parsing works out the grammatical relations between words: which noun is the subject of which verb, and so on. spaCy can draw it for you:

In [ ]:
from spacy import displacy

doc = nlp("A trifling incident thus served to settle a victory.")
displacy.render(doc, style="dep", jupyter=True, options={"distance": 90})

In [ ]:
# The same information as text: each token, its relation, and what it attaches to
for token in doc:
    print(f"{token.text:10s} {token.dep_:10s} -> {token.head.text}")

## Named entity recognition

This is the part most likely to be useful in a humanities project: finding mentions of **people, places and organisations** in a text.

Entities live in `doc.ents`, and each one carries a label (`PERSON`, `GPE` for countries and cities, `ORG`, `DATE`, and so on).

In [ ]:
def find_entities(text):
    """ return the named entities in a text, as a list of (text, label) tuples

    Args:
        text: the text to process, as a string

    Returns:
        A list of tuples, one per entity found
    """
    doc = nlp(text)
    return [(entity.text, entity.label_) for entity in doc.ents]


find_entities("In 1851 Charles Dickens travelled from London to Paris with the Royal Society.")

### Named entities on real historical text

Let's try it on the British Library books sample — nineteenth-century pages, digitised with OCR.

In [ ]:
!wget -q https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/main/Sessions/data/bl_books_sample.csv

In [ ]:
import pandas as pd

books = pd.read_csv("bl_books_sample.csv")

# Keep only pages where the OCR quality is reasonable:
books = books[books["mean_wc_ocr"] > 0.8]

page = books["text"].iloc[0]
print(page[:400])

In [ ]:
for entity, label in find_entities(page):
    print(f"{label:8s} {entity}")

Look at those results critically, because they are a good advertisement for scepticism.

On the page above, spaCy labels **Burgundy**, **Genoa** and **Marseilles** as `PERSON` when they are places; it reports `PERSON  Philip v^^and Richard`, where OCR damage has been passed straight through; and it makes `RICHARD` an organisation.

This is the central caveat for using NLP tools on historical material: **the model was trained on text that does not look like yours** — modern, clean, born-digital English. It will still be useful. But *how* useful is an empirical question about your corpus, and answering it means checking a sample by hand and reporting what you found.

## Applying spaCy to a whole dataframe

In notebook 3b you met `.apply()`, which runs a function over an entire column. That is how you scale any of this up:

In [ ]:
def count_people(text):
    """ how many PERSON entities are mentioned in this text? """
    doc = nlp(text)
    return len([entity for entity in doc.ents if entity.label_ == "PERSON"])


# Just the first 20 pages: spaCy is thorough, and therefore slow.
sample = books.head(20).copy()
sample["n_people"] = sample["text"].apply(count_people)

sample[["title", "n_people"]].head(10)

⚠️ **On speed.** spaCy runs a neural pipeline over every document, so it is orders of magnitude slower than counting words. Over thousands of documents, always: test on a handful first; disable the components you do not need (`spacy.load("en_core_web_sm", disable=["parser", "ner"])`); and only then run the whole corpus.

### ✏️ Exercise 2

Write a function that takes a text and returns the **places** mentioned in it (spaCy labels these `GPE` and `LOC`). Apply it to the first 20 pages of the books sample and add the result as a new column.

In [ ]:
# Type your code here:


# Solutions

### ✏️ Exercise 1

In [ ]:
def get_adjectives(text, nlp):
    """ return the adjectives in a text, as a list """
    doc = nlp(text)
    return [token.text for token in doc if token.pos_ == "ADJ"]


print(get_adjectives("It was a bright cold day in April, and the clocks were striking thirteen.", nlp))
print(get_adjectives("It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.", nlp))

### ✏️ Exercise 2

In [ ]:
def find_places(text):
    """ return the place names (GPE and LOC) mentioned in a text """
    doc = nlp(text)
    return [entity.text for entity in doc.ents if entity.label_ in ("GPE", "LOC")]


sample["places"] = sample["text"].apply(find_places)
sample[["title", "places"]].head(10)